# Private Analyst Fine-Tuning: A Safe Beginner Walkthrough

**Audience:** complete beginners who want to understand and safely operate this repository.

**Prerequisites:** Python 3.11 for GPU work, a terminal opened at the repository root, and no prior machine-learning experience.

**Learning goals:** inspect JSONL, validate reviewed examples, understand document-level splitting, preview commands, and recognize release evidence. Expensive or irreversible operations are disabled by default.

## Outline

1. Establish safe notebook state.
2. Build and validate a synthetic training example.
3. Split examples by document.
4. Inspect only aggregate private-data metadata.
5. Preview validation, training, evaluation, release, upload, and deployment commands.
6. Complete a review exercise.

In [ ]:
from __future__ import annotations

import hashlib
import json
import random
import subprocess
from collections import Counter
from pathlib import Path

SEED = 3407
ALLOW_PRIVATE_DATA = False
RUN_GPU_TRAINING = False
RUN_HUB_UPLOAD = False
RUN_DEPLOYMENT = False
random.seed(SEED)
{"seed": SEED, "private_data": ALLOW_PRIVATE_DATA, "gpu": RUN_GPU_TRAINING, "upload": RUN_HUB_UPLOAD, "deployment": RUN_DEPLOYMENT}

## 1. Find the repository root

A notebook may start from `notebooks/` or the repository root. This cell searches upward for `AGENTS.md` and does not read private data.

In [ ]:
def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "AGENTS.md").exists():
            return candidate
    raise FileNotFoundError("Open this notebook from inside the finetune repository.")

REPO_ROOT = find_repo_root(Path.cwd().resolve())
REPO_ROOT

## 2. Build one synthetic JSONL row

This example contains no corpus text. `messages` hold the conversation; `metadata` holds review and source-grouping controls.

In [ ]:
example = {
    "messages": [
        {"role": "system", "content": "You are a grounded financial analyst."},
        {"role": "user", "content": "Context:\nSynthetic lender margins fell.\n\nTask: Explain the risk."},
        {"role": "assistant", "content": "Lower margins reduce earnings resilience; the conclusion is limited to the supplied synthetic context."},
    ],
    "metadata": {"doc_id": "synthetic_report_001", "chunk_index": 0, "review_status": "approved"},
}
print(json.dumps(example, indent=2))

## 3. Validate the release-critical fields

Production training requires a non-empty assistant response, `metadata.doc_id`, and human approval.

In [ ]:
def validate_training_row(row: dict) -> list[str]:
    errors = []
    messages = row.get("messages")
    metadata = row.get("metadata")
    if not isinstance(messages, list):
        errors.append("messages must be a list")
    else:
        assistants = [message.get("content", "").strip() for message in messages if message.get("role") == "assistant"]
        if not any(assistants):
            errors.append("assistant response is empty")
    if not isinstance(metadata, dict) or not metadata.get("doc_id"):
        errors.append("metadata.doc_id is required")
    if not isinstance(metadata, dict) or metadata.get("review_status") != "approved":
        errors.append("metadata.review_status must be approved")
    return errors

assert validate_training_row(example) == []
"synthetic example passed"

## 4. Split by document

Rows from one report must stay together. This synthetic demonstration shows the same deterministic rule used by `finetune/train.py`.

In [ ]:
def choose_eval_documents(document_ids: list[str], fraction: float, seed: int) -> set[str]:
    unique_ids = sorted(set(document_ids))
    random.Random(seed).shuffle(unique_ids)
    count = min(len(unique_ids) - 1, max(1, round(len(unique_ids) * fraction)))
    return set(unique_ids[:count])

documents = ["report-a", "report-a", "report-b", "report-c", "report-c"]
eval_documents = choose_eval_documents(documents, 0.34, SEED)
train_documents = set(documents) - eval_documents
assert train_documents.isdisjoint(eval_documents)
{"train": sorted(train_documents), "eval": sorted(eval_documents)}

## 5. Optional private dataset inspection

This cell prints counts and hashes only. It never prints messages, filenames, document IDs, or source text. Change `ALLOW_PRIVATE_DATA` only when running locally in the trusted workspace.

In [ ]:
def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file_handle:
        for block in iter(lambda: file_handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

dataset_path = REPO_ROOT / "finetune" / "outputs" / "datasets" / "qwen35_approved_sft.jsonl"
if not ALLOW_PRIVATE_DATA:
    print("Skipped. Set ALLOW_PRIVATE_DATA=True only in the trusted local workspace.")
elif not dataset_path.exists():
    print(f"Not found: {dataset_path}")
else:
    rows = [json.loads(line) for line in dataset_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    statuses = Counter((row.get("metadata") or {}).get("review_status", "missing") for row in rows)
    documents = {(row.get("metadata") or {}).get("doc_id") for row in rows}
    print({"rows": len(rows), "documents": len(documents), "statuses": dict(statuses), "sha256": sha256(dataset_path)})

## 6. Preview the safe validation command

A command is an argument list: Python executable, script, then options. This cell displays it without running it.

In [ ]:
python_executable = REPO_ROOT / ".venv" / "Scripts" / "python.exe"
validation_command = [
    str(python_executable),
    str(REPO_ROOT / "finetune" / "train.py"),
    "--dry-run",
    "--dataset-path",
    str(dataset_path),
]
subprocess.list2cmdline(validation_command)

## 7. Optional GPU smoke run

Set `RUN_GPU_TRAINING=True` only after the approved dataset passes validation. This can consume significant GPU time.

In [ ]:
gpu_command = [
    str(python_executable), str(REPO_ROOT / "finetune" / "train.py"),
    "--dataset-path", str(dataset_path),
    "--output-dir", str(REPO_ROOT / "finetune" / "outputs" / "qwen35_4b_approved_smoke"),
    "--max-samples", "64", "--num-epochs", "0.1", "--skip-gguf-export",
]
if RUN_GPU_TRAINING:
    subprocess.run(gpu_command, cwd=REPO_ROOT, check=True)
else:
    print("Skipped GPU run:", subprocess.list2cmdline(gpu_command))

## 8. Evaluation and release commands

Evaluation creates machine-readable evidence. Release validation consumes that evidence and blocks incomplete bundles.

In [ ]:
run_dir = REPO_ROOT / "finetune" / "outputs" / "qwen35_4b_approved_v1"
benchmark_path = REPO_ROOT / "deployment" / "benchmarks" / "candidate.json"
release_command = [
    str(python_executable), str(REPO_ROOT / "finetune" / "validate_release.py"),
    "--run-dir", str(run_dir), "--benchmark-json", str(benchmark_path),
]
subprocess.list2cmdline(release_command)

## 9. Optional private upload and deployment

Upload requires a passed release manifest and `HF_TOKEN` in the environment. Deployment requires matching GGUF, mmproj, and checksum files. Both actions are disabled.

In [ ]:
upload_command = [
    str(python_executable), str(REPO_ROOT / "finetune" / "push_to_huggingface.py"),
    "--run-dir", str(run_dir),
    "--release-manifest", str(run_dir / "release_manifest.json"),
    "--repo-id", "YOUR_ACCOUNT/private-analyst-qwen35-v1",
]
deployment_command = [str(python_executable), str(REPO_ROOT / "deployment" / "bootstrap_local.py"), "--ingest-limit", "1024"]
if RUN_HUB_UPLOAD:
    subprocess.run(upload_command, cwd=REPO_ROOT, check=True)
else:
    print("Skipped upload:", subprocess.list2cmdline(upload_command))
if RUN_DEPLOYMENT:
    subprocess.run(deployment_command, cwd=REPO_ROOT, check=True)
else:
    print("Skipped deployment:", subprocess.list2cmdline(deployment_command))

## Exercise: review a rejected row

Predict the validation errors, run the cell, then repair the copy so it passes. Do not approve a real row merely to satisfy the validator; approval represents human quality review.

In [ ]:
exercise_row = json.loads(json.dumps(example))
exercise_row["messages"][-1]["content"] = ""
exercise_row["metadata"].pop("doc_id")
exercise_row["metadata"]["review_status"] = "draft"
errors = validate_training_row(exercise_row)
assert errors == ["assistant response is empty", "metadata.doc_id is required", "metadata.review_status must be approved"]
errors

## Finish

The safe path is: reviewed rows, document split, bf16 LoRA, assistant-only loss, baseline comparison, release gates, hashes, private publication, then verified deployment. See `docs/fine-tuning-visual-journal.md` for the complete commands and diagrams.